# 03 Â· Price-match requests and pricing recommendations

**SIMULATED** requests and competitor prices (no competitor website was scraped). Rules: `playbook/price_match_rules.md`, same as the Excel calculator.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")   # keep outputs clean
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # reuse the project's scripts
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
from common import OUT, SIM, RAW, PUBLIC
from simulate import decide
pm = pd.read_csv(SIM / 'price_match_requests.csv')
pm.shape

(1347, 31)

## 1. The rules on three requests

In [2]:
examples = pm.groupby('case_type_truth').head(1).set_index('case_type_truth')
cols = ['etg_price_eur', 'competitor_price_eur', 'etg_meal', 'competitor_meal', 'etg_cancellation', 'competitor_cancellation', 'competitor_taxes_included', 'net_cost_eur', 'alt_supplier_net_eur']
examples[cols]

,etg_price_eur,competitor_price_eur,etg_meal,competitor_meal,etg_cancellation,competitor_cancellation,competitor_taxes_included,net_cost_eur,alt_supplier_net_eur
case_type_truth,,,,,,,,,
like_for_like,1000.08,945.25,Breakfast,Breakfast,Non-refundable,Non-refundable,Yes,922.83,NaN
no_breakfast,121.05,112.73,Breakfast,Room only,Free cancellation,Free cancellation,Yes,110.48,102.41
taxes_excluded,322.42,303.78,Breakfast,Breakfast,Free cancellation,Free cancellation,No,294.63,NaN
old_screenshot,615.26,526.62,Breakfast,Breakfast,Free cancellation,Free cancellation,Yes,578.12,NaN
non_refundable,462.37,444.34,Breakfast,Breakfast,Free cancellation,Non-refundable,Yes,445.35,NaN
missing_evidence,249.18,211.98,Room only,Room only,Non-refundable,Non-refundable,Yes,234.64,NaN


In [3]:
examples.apply(decide, axis=1)

,adjusted_competitor_eur,valid_gap_eur,max_discount_eur,decision,discount_eur,reason
case_type_truth,,,,,,
like_for_like,945.25,54.83,38.80,Refuse,0.0,Valid gap but matching breaks the margin floor
no_breakfast,136.73,-15.68,5.97,Refuse,0.0,Not like-for-like: our rate includes breakfast
taxes_excluded,334.16,-11.74,15.51,Refuse,0.0,Not like-for-like: competitor price excludes ~...
old_screenshot,NaN,NaN,NaN,Ask for details,0.0,"Evidence incomplete (screenshot, age or dates)"
non_refundable,481.33,-18.96,0.00,Refuse,0.0,Not like-for-like: our rate is refundable (fle...
missing_evidence,NaN,NaN,NaN,Ask for details,0.0,"Evidence incomplete (screenshot, age or dates)"


## 2. Re-applying the rules to every logged request gives the logged decision

In [4]:
check = pm.apply(decide, axis=1)
(check['decision'] == pm['decision']).mean()

np.float64(1.0)

## 3. KPIs

In [5]:
for f in ['pm_summary', 'pm_by_decision', 'pm_by_case']:
    display(pd.read_csv(OUT / 'sim' / f'{f}.csv').round(3))

,requests,valid_like_for_like,valid_share,discount_share,refuse_share,switch_share,ask_share,discount_cost_eur,saved_sales_eur,median_handling_h,avg_claimed_gap
0,1347,787,0.584,0.21,0.512,0.108,0.169,2668.49,133877.43,2.2,0.079


,decision,requests,avg_handling_h,booked_rate,discount_cost_eur,share
0,Ask for details,228,1.137,0.469,0.00,0.169
1,Discount,283,3.069,0.827,3099.57,0.210
2,Refuse,690,3.312,0.355,0.00,0.512
3,Switch supplier,146,3.150,0.760,0.00,0.108


,case_type_truth,Ask for details,Discount,Refuse,Switch supplier
0,like_for_like,0,165,285,120
1,missing_evidence,131,0,0,0
2,no_breakfast,0,2,166,0
3,non_refundable,0,77,132,12
4,old_screenshot,97,0,0,0
5,taxes_excluded,0,39,107,14


## 4. Pricing recommendations (B2B hotel segments, last 90 days)
Elasticity range -1 to -3 is an assumption; the effect is shown as a range.

In [6]:
r = pd.read_csv(OUT / 'sim' / 'pricing_recommendations.csv')
r[['partner_type', 'destination_country', 'bookings', 'yoy_volume', 'margin_rate', 'valid_requests_per_1000', 'markup_change', 'margin_effect_eur_low', 'margin_effect_eur_high', 'recommendation']].round(3)

,partner_type,destination_country,bookings,yoy_volume,margin_rate,valid_requests_per_1000,markup_change,margin_effect_eur_low,margin_effect_eur_high,recommendation
0,API partner,AE,680,0.079,0.066,7.353,-0.005,-1303.839,-1134.051,Price pressure: -0.5 pt markup on this segment...
1,API partner,DE,710,0.160,0.059,7.042,0.000,-0.000,-0.000,Keep current markup; monitor
2,API partner,EL,678,0.119,0.056,7.375,0.000,0.000,0.000,Keep current markup; monitor
3,API partner,ES,1512,0.087,0.047,1.323,0.000,-0.000,-0.000,Hold price; margin close to floor -> renegotia...
4,API partner,FR,936,0.114,0.059,8.547,0.000,0.000,0.000,Keep current markup; monitor
5,API partner,IT,1223,0.075,0.061,4.088,0.000,0.000,0.000,Keep current markup; monitor
6,API partner,PT,616,0.023,0.063,6.494,0.000,0.000,0.000,Keep current markup; monitor
7,API partner,TR,923,0.115,0.054,3.250,0.005,1004.734,879.465,Low price pressure and strong growth: test +0....
8,Agency,AE,1198,0.138,0.092,1.669,0.005,2311.930,1816.686,Low price pressure and strong growth: test +0....
9,Agency,DE,1201,0.228,0.085,5.828,0.000,0.000,0.000,Keep current markup; monitor
